# Faster R-CNN: two-stage parameter experiments + interpretability (Google Colab, ~9.5 h GPU)

All runs train on **every 5th train frame** with one shared schedule (`EPOCHS`, `BATCH`, input size) and are
evaluated on every 5th val frame. Each run changes **one** parameter from the baseline
(`--rpn-proposals 2000`, `--roi-pos-fraction 0.25`, `--box-loss smooth_l1`):

| Run | Change | Parameter | Cost |
|---|---|---|---|
| `base` | none | baseline | 1 training run (~2.8 h) |
| `giou` | `--box-loss giou` | box regression loss (stage 2) | 1 training run |
| `pos05` | `--roi-pos-fraction 0.5` | RoI sampling ratio (stage 2) | 1 training run, only if time is left |
| `base` @ 300 / 1000 proposals | test-time RPN proposals | number of RPN proposals (stage 1) | eval only |

Then `src.frcnn_explain` looks inside the `base` model: where each GT object is lost (RPN vs RoI head),
a t-SNE of RoI features, and Grad-CAM over ResNet stages and FPN levels.

Everything is logged to **Weights & Biases** (one W&B run per model: config, loss curves per loss term,
val metrics, val frames with GT and predicted boxes, explain plots).

Before running:
- **Runtime -> Change runtime type -> T4 GPU** (or better).
- Add three secrets in Colab (**key icon in the left sidebar**, turn on notebook access for each):
  `WANDB_API_KEY`, `KAGGLE_USERNAME`, `KAGGLE_KEY` (from kaggle.com -> Settings -> API -> Create New Token).
  Accept the competition rules on Kaggle once, or the download fails.
- Checkpoints, eval logs and plots go to **Google Drive** (`MyDrive/frcnn_runs`), so a disconnect loses at most
  the epoch in progress: reconnect and **Run all** again, finished runs are skipped and the
  interrupted one continues from its `last.pt`.


In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount("/content/drive")
![ -d /content/repo/.git ] && git -C /content/repo pull -q || git clone -q https://github.com/LiliAlania/drone-multi-object-detection.git /content/repo
%cd /content/repo
!pip install -q pycocotools wandb scikit-learn kaggle

In [ ]:
import os, sys, time
from pathlib import Path

T0 = time.time()
REPO = Path("/content/repo")
sys.path.insert(0, str(REPO))
MOD = "src.models.faster_rcnn"
assert (REPO / "src/models/faster_rcnn.py").is_file(), "push the src/models/ layout to GitHub first"
# the code comes from GitHub, not from this notebook: stop now if it predates W&B logging and frcnn_explain
assert "--wandb-project" in (REPO / "src/models/faster_rcnn.py").read_text() and (REPO / "src/frcnn_explain.py").is_file(), \
    "GitHub has an older version of the repo: commit and push src/ before running this notebook"

EXPERIMENTS = {
    "base": "",
    "giou": "--box-loss giou",
    "pos05": "--roi-pos-fraction 0.5",  # runs only if the time budget allows
}
RUN = ["base", "giou", "pos05"]
PROPOSALS = [300, 1000]  # test-time sweep on `base`; its own eval is the 2000 point

EPOCHS = 5         # step LR drops after epochs 3 and 4; lower it if one epoch takes > 35 min
SUBSET_STEP = 5    # every 5th train frame: ~4.8k images, one epoch ~32 min on a T4
BATCH = 4          # 2 if CUDA out of memory
EVAL_STEP = 5      # every 5th val frame, as in the README
BUDGET_MIN = 690   # whole session; free Colab stops at 12 h at most, often sooner
RESERVE_MIN = 60   # kept free for the proposals sweep, the explain cells and the summary
RUNS = Path("/content/drive/MyDrive/frcnn_runs")  # on Drive: survives disconnects

# W&B and Kaggle: keys from Colab secrets; the `!python` subprocesses inherit it through the environment
WANDB_PROJECT = "drone-multi-object-detection"
WANDB_ENTITY = "irynkasv-ukrainian-catholic-university-org"  # your W&B username or a team you can write to; empty = the key's default entity
if WANDB_ENTITY:
    os.environ["WANDB_ENTITY"] = WANDB_ENTITY
os.environ["WANDB_DIR"] = str(RUNS)  # offline runs then land on Drive
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name)
    except Exception:  # missing, or notebook access not granted
        return None

for name in ["KAGGLE_USERNAME", "KAGGLE_KEY"]:  # the kaggle CLI reads these env vars
    if secret(name):
        os.environ[name] = secret(name)
if secret("WANDB_API_KEY"):
    os.environ["WANDB_API_KEY"] = secret("WANDB_API_KEY")
else:  # log offline instead of crashing; upload later with `wandb sync`
    os.environ["WANDB_MODE"] = "offline"
    print(f"No WANDB_API_KEY secret: W&B logs offline into {RUNS}/wandb")
RUNS.mkdir(parents=True, exist_ok=True)

In [ ]:
# competition data: downloaded once per Colab session into local disk (fast reads), not Drive
INPUT = Path("/content/data")
find = lambda: [p for depth in range(0, 5) for p in INPUT.glob("*/" * depth + "val/annotations_coco.json")]
if not find():
    assert os.environ.get("KAGGLE_KEY"), "add the KAGGLE_USERNAME and KAGGLE_KEY secrets first"
    !python -m src.get_data --output-dir {INPUT}
found = find()
assert found, "Competition data not found after download"
VAL_ROOT = found[0].parent
DATA = VAL_ROOT.parent
print("data:", DATA, sorted(p.name for p in DATA.iterdir()))

SUBSET = Path("/content/visdrone-subset")
if not (SUBSET / "images" / "train").is_dir():
    !python -m src.make_subset --data-root {DATA} --output-dir {SUBSET} --step {SUBSET_STEP} --splits train val
    !rm -f {SUBSET}.zip
print(f"{len(list((SUBSET / 'images' / 'train').rglob('*.jpg')))} train images, "
      f"{(time.time() - T0) / 60:.0f} min elapsed")

In [ ]:
run_minutes = []  # wall time of finished runs, to decide whether the next one still fits
done = []
for name in RUN:
    out = RUNS / f"frcnn_{name}"
    if "mAP@" in ((out / "eval.txt").read_text(errors="ignore") if (out / "eval.txt").is_file() else ""):
        print(f"{name}: already trained and evaluated in an earlier session")
        done.append(name)
        continue
    elapsed = (time.time() - T0) / 60
    if run_minutes and elapsed + max(run_minutes) > BUDGET_MIN - RESERVE_MIN:
        print(f"skip {name}: {elapsed:.0f} min used, a run takes ~{max(run_minutes):.0f} min")
        continue
    flags = EXPERIMENTS[name]
    resume = "--resume" if (out / "last.pt").is_file() else ""  # after a disconnect, continue the interrupted run
    print(f"=== {name}: {flags or 'baseline'} ===")
    start = time.time()
    !python -m {MOD} train --data-root {SUBSET} --epochs {EPOCHS} --batch {BATCH} --workers 2 {flags} --out-dir {out} {resume} --wandb-project {WANDB_PROJECT}
    !python -m {MOD} evaluate --weights {out}/last.pt --split-root {VAL_ROOT} --batch {BATCH} --step {EVAL_STEP} --wandb-project {WANDB_PROJECT} | tee {out}/eval.txt
    run_minutes.append((time.time() - start) / 60)
    done.append(name)
    print(f"{name}: {run_minutes[-1]:.0f} min, {(time.time() - T0) / 60:.0f} min elapsed")

In [ ]:
# RPN proposals at test time: same `base` weights, only the number of proposals passed to the RoI head changes
import importlib
import torch

frcnn = importlib.import_module(MOD)

def with_proposals(n):
    # the checkpoint's saved model_cfg holds rpn_proposals=2000; override it after it is passed in
    def build(*args, **kwargs):
        return frcnn.build_model(*args, **{**kwargs, "rpn_proposals": n})
    return build

sweep = []
for n in PROPOSALS:
    print(f"=== base @ {n} proposals ===")
    metrics = frcnn.evaluate(RUNS / "frcnn_base" / "last.pt", VAL_ROOT, BATCH, EVAL_STEP, build_fn=with_proposals(n))
    sweep.append({"run": "base", "change": f"{n} RPN proposals (test)", "rpn_proposals": n, **metrics})
    torch.cuda.empty_cache()

In [ ]:
# summary: mAP, AP50, AP_small and inference speed per run
import re
import pandas as pd
import wandb

rows = []
for name in done:
    text = (RUNS / f"frcnn_{name}" / "eval.txt").read_text(errors="ignore")
    ap = re.search(r"mAP@\[\.5:\.95\] ([\d.]+) \| AP50 ([\d.]+) \| AP_small ([\d.]+)", text)
    ms = re.search(r"Inference: ([\d.]+) ms/image", text)
    if not ap:
        print(f"{name}: no metrics in eval.txt")
        continue
    rows.append({"run": name, "change": EXPERIMENTS[name] or "baseline", "rpn_proposals": 2000,
                 "mAP": float(ap[1]), "AP50": float(ap[2]), "AP_small": float(ap[3]),
                 "ms_per_image": float(ms[1]) if ms else None})

cols = ["run", "change", "rpn_proposals", "mAP", "AP50", "AP_small", "ms_per_image"]
summary = pd.DataFrame(rows + sweep)[cols]
summary.to_csv(RUNS / "frcnn_experiments.csv", index=False)
print(f"settings: epochs {EPOCHS}, subset step {SUBSET_STEP}, batch {BATCH}, eval step {EVAL_STEP}")

# one comparison table across all runs in the W&B project
try:
    run = wandb.init(project=WANDB_PROJECT, name="frcnn_experiments_summary", job_type="summary", config={"epochs": EPOCHS, "subset_step": SUBSET_STEP, "batch": BATCH, "eval_step": EVAL_STEP})
    run.log({"frcnn_experiments": wandb.Table(dataframe=summary)})
    run.finish()
except Exception as error:  # W&B access problems must not hide the table below
    print("W&B summary not logged:", error)
summary.round(4)

## What the base model learns and where it fails

`objects` (every 5th val frame): for each GT object, the stage that lost it (no RPN proposal / proposal but
score < 0.3 / wrong class), what the RoI head says when given the exact GT box, and a t-SNE of the 1024-d RoI
features coloured by class, size and outcome. `gradcam` (every 50th val frame, 2 TP + 2 FP per frame):
Grad-CAM of single detection scores on C2..C5 and P2..P5. Plots land in `runs/frcnn_base/explain/` and in the
`frcnn_base` W&B run.

In [ ]:
from IPython.display import Image, display

BASE = RUNS / "frcnn_base" / "last.pt"
!python -m src.frcnn_explain objects --weights {BASE} --split-root {VAL_ROOT} --step {EVAL_STEP} --wandb-project {WANDB_PROJECT}
!python -m src.frcnn_explain gradcam --weights {BASE} --split-root {VAL_ROOT} --step 50 --wandb-project {WANDB_PROJECT}
for name in ["outcomes", "head_confusion", "tsne", "gradcam_levels", "gradcam_focus", "gradcam_gallery"]:
    display(Image(filename=str(BASE.parent / "explain" / f"{name}.png")))
print(f"{(time.time() - T0) / 60:.0f} min elapsed")